# Options Pricer & Greeks Analyzer — Synthèse

*Notebook de synthèse — lecture en 5 minutes. Le détail complet (dérivations, code, validations pas-à-pas) est dans les notebooks `03` à `08`.*

**Le projet en 3 phrases.** Ce projet implémente de bout en bout le pricing d'options européennes sous Black-Scholes — prix analytiques, Greeks du 1er et du 2nd ordre, validation croisée par Monte Carlo — avant de confronter le modèle à des données de marché réelles (SPY, via `yfinance`) pour en révéler la limite centrale : la volatilité n'est pas constante. Il se termine par la simulation d'une couverture delta-neutre discrète et la décomposition analytique du P&L qui en résulte (Gamma, Theta, intérêts, résidu de vol). Chaque brique est validée indépendamment — parité call-put, différences finies, round-trip Newton-Raphson, intervalle de confiance Monte Carlo — et couverte par **84 tests pytest**.

**Arc narratif.**

1. **Pricer** un call/put européen analytiquement, validé par Monte Carlo.
2. **Mesurer les risques** avec les Greeks — direction (Δ), convexité (Γ), volatilité (Vega, Vanna, Volga), temps (Θ, Charm).
3. **Montrer où le modèle casse** : sur données réelles, la vol implicite dépend du strike et de la maturité (skew, term structure) — l'hypothèse de vol constante de Black-Scholes est empiriquement fausse.
4. **Couvrir et expliquer le P&L** : simuler une couverture delta-neutre rebalancée périodiquement et décomposer le P&L résultant terme à terme.

---

## 1. Pricing — Black-Scholes analytique

Le prix d'un call/put européen se calcule en forme fermée sous l'hypothèse de mouvement brownien géométrique. La cohérence interne du modèle est vérifiée par la **parité call-put** ($C - P = S - Ke^{-rT}$), qui doit être exacte à la précision machine près puisqu'elle ne dépend d'aucune approximation numérique. Le prix est ensuite validé de façon indépendante par simulation Monte Carlo (notebook `04`), qui confirme le résultat analytique par une méthode entièrement différente.

**Résultat clé : parité call-put vérifiée avec un écart < 1e-9** (C=10.4506, P=5.5735, S=K=100, T=1, r=5%, σ=20%).

![Prix vs Spot](../figures/01_price_vs_spot.png)

Le prix BS domine toujours le payoff à maturité (valeur temps > 0), maximale ATM et nulle deep ITM/OTM — la convergence vers les droites de payoff aux extrêmes confirme la cohérence du modèle.

---

## 2. Greeks — 1er et 2nd ordre

Les Greeks du premier ordre (Δ, Γ, Vega, Θ, ρ) mesurent la sensibilité du **prix** aux paramètres de marché ; ceux du second ordre (Vanna, Volga, Charm) mesurent la sensibilité de ces sensibilités — comment le **hedge lui-même** se déforme quand le marché bouge. Les formules fermées du second ordre sont validées par **différences finies centrées** sur les Greeks du premier ordre (ex. Volga ≈ [Vega(σ+h) − Vega(σ−h)] / 2h), une vérification indépendante de toute dérivation analytique.

**Résultat clé : écarts différences finies vs formules analytiques < 1e-4** sur les trois Greeks du second ordre (Vanna, Volga, Charm).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
import pandas as pd
from greeks import all_greeks

S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20

table = pd.DataFrame({
    "Call": all_greeks(S, K, T, r, sigma, "call", include_second_order=True),
    "Put":  all_greeks(S, K, T, r, sigma, "put",  include_second_order=True),
}).T
table.index.name = "Option"
table.round(5)

,Delta,Gamma,Vega,Theta,Rho,Vanna,Volga,Charm
Option,,,,,,,,
Call,0.63683,0.01876,0.37524,-0.01757,0.53232,-0.28143,9.85006,-0.06567
Put,-0.36317,0.01876,0.37524,-0.00454,-0.41890,-0.28143,9.85006,-0.06567


![Gamma et Vega vs Spot](../figures/03_gamma_vega_vs_spot.png)

Gamma et Vega forment une cloche centrée ATM : c'est là que la couverture est la plus coûteuse à maintenir. Un vendeur d'options est structurellement short Gamma et short Vega.

![Volga vs Spot](../figures/12_volga_vs_spot.png)

Le Volga (convexité en vol) est nul ATM et positif sur les deux ailes OTM/ITM — c'est le lien direct avec le smile : le marché surpaye les options loin de la monnaie pour couvrir ce coût de convexité en volatilité.

---

## 3. Volatilité implicite & surface

Black-Scholes n'a pas d'inverse analytique pour σ : la vol implicite s'obtient par inversion numérique (**Newton-Raphson**, avec repli sur **Brent** si la convergence échoue). Le test de cohérence le plus direct est le **round-trip** : pricer avec un σ connu, inverser le prix obtenu, et vérifier qu'on retrouve exactement le σ de départ. Appliqué sur la chaîne d'options SPY réelle, ce même solveur produit un **skew** net (les puts OTM cotent une vol bien supérieure aux calls OTM) et une **term structure** — la preuve empirique que l'hypothèse de vol constante de Black-Scholes est fausse.

**Résultat clé : round-trip Newton-Raphson — écart de 0.00e+00** entre σ vrai et σ retrouvé (call et put, à partir du prix théorique σ=20%).

![Volatility Smile SPY](../figures/07_volatility_smile.png)

Skew net sur SPY : IV ≈ 66 % pour les puts OTM profonds ($K/S \approx 0.67$) contre ≈ 12–14 % pour les calls OTM — la prime de risque de krach que Black-Scholes, à volatilité unique, ne peut pas reproduire.

![Surface de volatilité 3D](../figures/08_vol_surface.png)

La surface complète (6 maturités × 40 points de moneyness, IV entre 12 % et 37 %) combine le skew transversal et la term structure longitudinale en une seule carte du risque de volatilité.

---

## 4. Couverture delta-neutre & P&L

La couverture d'un call vendu est simulée sur une trajectoire GBM discrétisée en $n$ pas : à chaque rebalancement, la position en actions annule l'exposition directionnelle, et le P&L incrémental se décompose par développement de Taylor en coût Gamma, gain Theta, intérêts sur cash, et résidu de vol (écart vol réalisée / vol implicite). Deux résultats structurent cette section : l'erreur de réplication avec rebalancement discret décroît en $\sqrt{T/n}$, et le vendeur d'option est en espérance un **vendeur de variance** — il gagne si le marché réalise moins de volatilité que l'implicite, et perd dans le cas contraire.

**Résultats clés :**
- Ratio empirique std(hebdo)/std(quotidien) **≈ 2.15**, conforme au ratio théorique $\sqrt{252/52} \approx 2.20$.
- Le **point mort du P&L moyen se situe exactement à $\sigma_{\text{réal}} = \sigma_{\text{impl}}$**.

![Fréquence de rebalancement](../figures/17_hedge_frequency.png)

La couverture quotidienne ($n=252$, std ≈ 0.45) est nettement plus resserrée que l'hebdomadaire ($n=52$, std ≈ 1.0) — confirmation empirique de la décroissance en $\sqrt{T/n}$.

![P&L vs vol réalisée](../figures/18_vol_mismatch_pnl.png)

Le P&L moyen du vendeur décroît linéairement avec $\sigma_{\text{réal}}$ et croise zéro exactement à $\sigma_{\text{réal}} = \sigma_{\text{impl}} = 20\%$ — le point mort théorique attendu de la décomposition Taylor.

---

## 5. Validation

Chaque brique du projet est vérifiée par une méthode indépendante de son implémentation :

| Brique | Méthode de validation | Critère |
|--------|------------------------|---------|
| Pricing Black-Scholes | Parité call-put $C-P=S-Ke^{-rT}$ | écart < 1e-9 |
| Pricing Monte Carlo | Intervalle de confiance à 95 % autour du prix BS | prix BS ∈ IC 95 % |
| Greeks 2nd ordre (Vanna, Volga, Charm) | Différences finies centrées sur les Greeks 1er ordre | écart < 1e-4 |
| Solveur de vol implicite | Round-trip prix → σ → prix | écart σ < 1e-6 (mesuré : 0.00e+00) |
| Convergence Monte Carlo | Régression log-log erreur vs N | pente ≈ −0.5 (théorique : −0.5) |
| Couverture delta-hedge | Ratio std(hebdo)/std(quotidien) | conforme à $\sqrt{252/52}$ |

Ces vérifications sont automatisées dans **84 tests pytest** (`tests/test_black_scholes.py`, `test_greeks.py`, `test_monte_carlo.py`, `test_implied_vol.py`, `test_delta_hedge.py`) :

```bash
pytest tests/ -v
```

---

## 6. Limites & extensions

**Limites du modèle actuel**

- Volatilité constante : Black-Scholes ne peut pas reproduire le skew observé avec un seul σ — cf. section 3.
- Options européennes uniquement (pas d'exercice anticipé).
- Taux sans risque déterministe et constant, absence de dividendes.
- Couverture simulée sans coûts de transaction : chaque rebalancement est supposé gratuit.

**Extensions envisagées**

- **Modèle de Heston** : volatilité stochastique à retour à la moyenne ($dV = \kappa(\theta - V)dt + \xi\sqrt{V}dW$), pour reproduire le skew et le smile endogènement plutôt que de les mesurer a posteriori.
- **Options américaines** : algorithme de Longstaff-Schwartz (Monte Carlo avec régression) pour l'exercice anticipé.
- **Coûts de transaction dans le hedge** : introduire un coût proportionnel au volume rebalancé pour arbitrer réalistement fréquence de couverture et erreur de réplication.
- **Modèle à sauts de Merton** pour capturer les queues épaisses observées sur le marché.

---

*Projet complet : [`README.md`](../README.md) · Code source : [`src/`](../src/) · Notebooks détaillés : [`notebooks/`](.) · Tests : [`tests/`](../tests/)*